# demo_b3

> Phase 22 (`specs/multiagent/phase22/`, experiment B3): a better cheap screen. Quick (a tiny grid fit, a fake experiment's forecast and score, the pooled gate, report and figure, in this process), then the week-1 fit and the retrospective check, and an experiment's forecast, run by `tools/b3_run.py` (multiprocess, a `__main__` guard) and read here behind `EXPLRL_B3_STAGE`.

Artifacts (gitignored): `results/multiagent/b3quick/` (quick) and `results/multiagent/b3/` (`grid/`, `choose.json`, `fit.json`, `retro.json`, `retro.md`, `report.md`, `score.json`, `b3.png`). Forecast and score files are committed beside each experiment's spec (`specs/multiagent/phase20/c1_predictions.json`, `c1_score.json`). B3 makes no API calls and uses no GPU (NFR-2).

- **Quick** (normal tests): everything below on a few 8-agent boards.
- **Fit** (`--flags slow`, `EXPLRL_B3_STAGE=fit`): after `tools/b3_run.py check`, `grid`, `fit` and `retro` (T4, T5; about 1.5 hours on 8 workers), reads `fit.json` and `retro.md`. Needs `B3_PLAN` frozen and pushed; stop and ask on failed fit health (P22-11).
- **Forecast** (`--flags slow`, `EXPLRL_B3_STAGE=forecast`): after `tools/b3_run.py forecast C1` (T6, once C1's pilot rows exist), reads `c1_predictions.json` and prints its SHA-256 for `C1_PLAN`.

In [ ]:
import hashlib
import json
import os
import tempfile
from pathlib import Path

from explRL.b3_cells import CELL, thetaKey
from explRL.b3_fit import chooseKnob, fit, gridSim, refit
from explRL.b3_forecast import Effect, forecastExperiment, scoreExperiment, writeForecast
from explRL.b3_report import figB3, gateB3, poolScores, syntheticB3, writeB3
from explRL.b3_retro import retroCellFn
from explRL.train import demo_artifact_dir

STAGE = os.environ.get('EXPLRL_B3_STAGE', 'quick')
B3 = demo_artifact_dir(None, 'multiagent/b3')
REPO = B3.parents[2]

## Quick

In [ ]:
# quick (FR-D6): a tiny fit, a fake experiment's five lines and its score, then the pooled gate, report and figure
q = demo_artifact_dir('b3quick', 'multiagent')
tiny = [CELL[c] for c in ('colony/stage1:colony|independent:8', 'colony/stage1:plain|clean:8')]
g = [dict(trustOne=a, spread=b) for a in (0.0, 1.0) for b in (0.0, 1.0)]
sims = gridSim(tiny, g, workers=0, seeds=range(3000, 3003), out=Path(tempfile.mkdtemp()))
T = {c['id']: (sims['trustOne=1.00'][c['id']] + 0.01, 0.02, 8) for c in tiny}
th0 = fit(list(T), sims, T)
assert thetaKey(th0) == 'trustOne=1.00'
fx = (Effect('F-1', 'fake', 'honestPooled', 'independent', 8, ((1, 'colonyGuards'), (-1, 'colony'))),
      Effect('F-2', 'fake', 'contamination', 'independent', 8, ((1, 'plain'), (-1, 'colony'))),
      Effect('F-3', 'fake', 'honestPooled', 'independent', 8, ((1, 'record'), (-1, 'colony')), represented=False))
pilot = [dict(level='colony|independent', N=8, seed=1890, honestPooled=0.10, contamination=0.5),
         dict(level='colonyGuards|independent', N=8, seed=1890, honestPooled=0.25, contamination=0.2)]
f = forecastExperiment('FAKE', fx, retroCellFn, pilot, [], [], (), seeds=range(3000, 3004), nboot=200,
                       fitFn=lambda rd: th0, tauFn=lambda th, rd: dict(trustOne=0.1), refitFn=lambda th, tau, p: th)
assert set(f['items']['F-1']['lines']) == {'fitted', 'priorOnly', 'unfitted', 'pilotOnly', 'naive'}
assert set(f['items']['F-3']['lines']) == {'pilotOnly', 'naive'} and f['items']['F-3']['lines']['pilotOnly']['fallback']
sha = writeForecast(f, q / 'fake_predictions.json')
assert sha == hashlib.sha256((q / 'fake_predictions.json').read_bytes()).hexdigest()
s = scoreExperiment(q / 'fake_predictions.json', {'F-1': (0.1, [0.05, 0.15]), 'F-2': (0.3, [0.2, 0.4]), 'F-3': (0.0, [-0.1, 0.1])})
assert s['n'] == dict(common=2, all=3, pilotComputed=1)   # F-2 lacks plain in the pilot: fallback and (q / 'fake_score.json').exists()
paths, fitRes, retro = syntheticB3(q)
P = poolScores(paths)
G = gateB3(P)
writeB3(fitRes, retro, P, G, q)
figB3(P, retro, q / 'b3.png')
# AC-9: tools/b3_run.py grid refuses an unfrozen B3_PLAN (before any board is played)
import subprocess, sys
from explRL.b3_plan import B3_PLAN
if not B3_PLAN['frozen']:
    r = subprocess.run([sys.executable, str(REPO / 'tools/b3_run.py'), 'grid', '1'], capture_output=True, text=True)
    assert r.returncode != 0 and 'freeze and push B3_PLAN' in r.stderr, r.stderr[-500:]
print(G['verdict'], G['right'])
print({k: v['lines'] for k, v in f['items'].items()})

## Week-1 fit and the retrospective check (T4, T5)

In [ ]:
#| slow
if STAGE != 'fit':
    print('skipped: needs EXPLRL_B3_STAGE=fit after tools/b3_run.py check, grid, fit and retro')
else:
    F = json.loads((B3 / 'fit.json').read_text())
    print('knob', F['knob'], 'fits', F['fits'], 'tau', F['tau'], 'loss', F['loss'])
    print('fit health', 'ok' if F['health']['ok'] else 'STOP AND ASK (P22-11)', F['health'])
    print((B3 / 'retro.md').read_text())

## A forecast (T6)

In [ ]:
#| slow
if STAGE != 'forecast':
    print('skipped: needs EXPLRL_B3_STAGE=forecast after tools/b3_run.py forecast C1')
else:
    p = REPO / 'specs/multiagent/phase20/c1_predictions.json'
    C = json.loads(p.read_text())
    print('SHA-256 for C1_PLAN:', hashlib.sha256(p.read_bytes()).hexdigest(), 'commit', C['commit'], 'B3_PLAN', C['b3Plan'])
    print('theta', C['theta'], 'tau', C['tau'])
    for i, it in C['items'].items():
        print(i, {k: v['sign'] for k, v in it['lines'].items()})